# Weather Trend Analysis and Extreme Temperature Pattern Detection

Interactive walkthrough of the end-to-end workflow: NOAA GHCN-Daily ingestion, feature engineering, seasonal-baseline labeling, time-aware baseline models, and evaluation. Run `python scripts/run_pipeline.py` first (or this notebook) to populate the `reports/` artifacts.

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
SRC = os.path.join(PROJECT_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from weather_extremes.config import load_config
from weather_extremes.pipeline import load_dataset, prepare_features
from weather_extremes.model import build_models, time_split, fit_model, predict_model, feature_importance
from weather_extremes.evaluate import evaluate_model, season_error_breakdown

sns.set_theme(style='whitegrid')
config = load_config()
print(config['data']['station_id'], config['data']['start_date'], '->', config['data']['end_date'])

## 1. Data foundation - NOAA GHCN-Daily
Downloads the quality-reviewed daily summaries for the selected station and aligns them to a continuous daily index (TMAX, TMIN, PRCP, SNOW, SNWD).

In [ ]:
wide, station = load_dataset(config)
print(station)
wide.tail()

## 2. Feature engineering
Cleaning, calendar/cyclical signals, causal rolling statistics, lagged temperatures, and seasonal-baseline anomalies. All features are strictly past-only to avoid leakage.

In [ ]:
frame, feature_cols, thresholds = prepare_features(config, wide)
print('rows:', len(frame))
print('features:', len(feature_cols))
feature_cols

## 3. Labelling extremes
Observations are compared with the seasonal (day-of-year) baseline. Days above the hot quantile are `extreme_hot`, below the cold quantile are `extreme_cold`, and the rest are `normal`.

In [ ]:
print(json.dumps(thresholds['class_counts'], indent=2))
frame['target_name'].value_counts()

## 4. Trend analysis
Annual mean temperatures and the long-term trend (reported in `reports/metrics/run_summary.json`).

In [ ]:
annual = frame.groupby('year')[['TMAX', 'TMIN']].mean()
annual.plot(figsize=(12, 4), marker='o', title='Annual mean temperature')
plt.ylabel('deg C')
plt.tight_layout()
plt.show()

## 5. Time-aware split and baseline models
Chronological train/validation/test split (no shuffling) and baseline classifiers: logistic regression and random forest.

In [ ]:
train, val, test = time_split(frame, config['split']['train_frac'], config['split']['val_frac'], config['split']['test_frac'])
print('train/val/test:', len(train), len(val), len(test))

models = build_models(config)
predictions = {}
for name, model in models.items():
    fit_model(model, train, feature_cols)
    pred = predict_model(model, test, feature_cols)
    predictions[name] = pred
    m = evaluate_model(test['target'], pred, config['evaluation']['alert_threshold'])
    auc = m['roc_auc_extreme_vs_normal']
    print(f"{name}: acc={m['accuracy']:.3f} f1_macro={m['f1_macro']:.3f} f1_extreme={m['f1_extreme_macro']:.3f} auc_extreme={auc:.3f}")

## 6. Error breakdown and interpretation
Which seasons generate the most false alarms, and which signals drive the detector.

In [ ]:
name = config['model']['primary']
breakdown = season_error_breakdown(test['season'], test['target'], predictions[name]['pred'], predictions[name]['p_extreme'])
breakdown

In [ ]:
importance = feature_importance(models[name], feature_cols)
importance.head(15)

## 7. Next steps
- Broaden to multi-station and regional comparisons.
- Calibrated thresholds, class balancing and stronger time-series models.
- Explainability, uncertainty bands and drift monitoring.
- Package as a reproducible dashboard / alerting tool.

All figures are written to `reports/figures/` and metrics to `reports/metrics/`.